# Workflow 1: Identify Native Non-Covalent Lasso Entanglements (NCLEs)

## Goal

Identify all native NCLEs present in a reference protein structure, filter for high-confidence entanglements, cluster redundant variants, and compute structural features for the representative set.

---

## Typical runtime

| Step | Runtime |
|------|---------|
| Native entanglement detection | 1–10 minutes |
| High-quality filtering | < 1 minute |
| Clustering | < 1 minute |
| Feature generation | 1–5 minutes |

---

## Required input files

| File | Path | Role |
|------|------|------|
| Cleaned all-atom PDB | `$DATASTORE/user_input/reference_structures/1zmr_model_clean.pdb` | Input structure |
| Cα PSF topology | `$DATASTORE/user_input/reference_structures/1zmr_model_clean_ca.psf` | Topology reference |
| Cα COR coordinates | `$DATASTORE/user_input/reference_structures/1zmr_model_clean_ca.cor` | Coordinate reference |

> **Note:** This tutorial demonstrates analysis on an all-atom structure, but EntDetect can also handle Cα coarse-grained structures.

---

> **Environment:** Before running this notebook, activate the `entdetect` conda environment and register it as a kernel:
> ```bash
> conda activate entdetect
> python -m ipykernel install --user --name entdetect --display-name "entdetect"
> ```
> Then select **entdetect** from the kernel picker (top-right of VS Code or Jupyter).

## Step 1. Set up paths

Set the DATASTORE root and create all output directories needed for this workflow.  
*(The `conda activate` command is handled outside the notebook — see the environment note above.)*

In [ ]:
import os

DATASTORE = "/scratch/ims86/EntDetect_Datastore"
# OUTDIR    = f"{DATASTORE}/outputs/workflow1"
OUTDIR    = f"{DATASTORE}/tmp/outputs/workflow1"

os.makedirs(f"{OUTDIR}/Native_GE",                    exist_ok=True)
os.makedirs(f"{OUTDIR}/Native_HQ_GE",                 exist_ok=True)
os.makedirs(f"{OUTDIR}/Native_clustered_HQ_GE",       exist_ok=True)
os.makedirs(f"{OUTDIR}/Native_clustered_HQ_GE_features", exist_ok=True)

print(f"DATASTORE : {DATASTORE}")
print(f"OUTDIR    : {OUTDIR}")
print("Output directories ready.")

## Step 2. Prepare a cleaned structure

The `1zmr_model_clean.pdb` file in `$DATASTORE/user_input/reference_structures/` is already cleaned and ready to use. For your own protein you would:

- rebuild missing residues and atoms (e.g., with **Modeller** or **CHARMM**);
- remove duplicate residue records;
- verify chain labels and residue numbering are sensible;
- ensure the PDB contains only the protein chain(s) of interest.

## Step 3. Detect native entanglements

Scans the input structure and identifies all **native non-covalent lasso entanglements** present in the reference conformation. Each NCLE is characterized by a loop closed by a disulfide or non-covalent contact and one or more chain termini that thread through it.

**Practical notes:**
- All-atom PDB (conventional): `Calpha=False`, `CG=False`
- All-atom but use Cα contacts: `Calpha=True`, `CG=False`
- Cα-only coarse-grained model: `Calpha=True`, `CG=True`
- `g_threshold=0.6` is the standard cutoff. Lower values detect more (but potentially noisier) entanglements.
- `ent_detection_method=2`: any nonzero TLN for either termini (tutorial default). Use `3` for production (both GLN and TLN nonzero for same termini).

**Output:** `$OUTDIR/Native_GE/1ZMR_GE.csv`

In [ ]:
from EntDetect.gaussian_entanglement import GaussianEntanglement

# ── Inputs ──────────────────────────────────────────────────────────────────
pdb           = f"{DATASTORE}/user_input/reference_structures/1zmr_model_clean.pdb"
native_outdir = f"{OUTDIR}/Native_GE"
ID            = "1ZMR"

# ── Initialize ───────────────────────────────────────────────────────────────
ge = GaussianEntanglement(g_threshold=0.6, density=0.0, Calpha=False, CG=False,
                          ent_detection_method=2)

# ── Run ──────────────────────────────────────────────────────────────────────
ge.calculate_native_entanglements(pdb_file=pdb, outdir=native_outdir, ID=ID, chain='A')

In [ ]:
import pandas as pd

result_step3 = pd.read_csv(f"{native_outdir}/1ZMR_GE.csv", sep='|')
print(f"Detected entanglements: {len(result_step3)}")
result_step3.head()

## Step 4. Filter for high-quality entanglements

This step is especially important when using **AlphaFold** or other predicted structures, or when you want to remove slipknots and low-confidence entanglements.

| `model` value | Use when |
|---------------|----------|
| `"EXP"` | X-ray crystallography, cryo-EM, or NMR |
| `"AF"` | AlphaFold prediction |

> **Note (non-canonical regions):** If the structure contains expression tags, fusion segments, or engineered regions, a mapping file can be supplied to restrict the filtered set to the canonical region only. See `select_high_quality_entanglements` documentation for the expected format.

**Output:** `$OUTDIR/Native_HQ_GE/1ZMR.csv`

In [ ]:
from EntDetect.gaussian_entanglement import GaussianEntanglement

pdb              = f"{DATASTORE}/user_input/reference_structures/1zmr_model_clean.pdb"
native_HQ_outdir = f"{OUTDIR}/Native_HQ_GE"
ID               = "1ZMR"

# Point to Step 3 output
NCLE_file = f"{OUTDIR}/Native_GE/1ZMR_GE.csv"

ge = GaussianEntanglement(g_threshold=0.6, density=0.0, Calpha=False, CG=False,
                          ent_detection_method=2)

ge.select_high_quality_entanglements(
    NCLE_file,
    pdb,
    outdir=native_HQ_outdir,
    ID=ID,
    model="EXP",   # "EXP" for experimental structures; "AF" for AlphaFold models
    chain='A'
)

In [ ]:
result_step4 = pd.read_csv(f"{native_HQ_outdir}/1ZMR.csv", sep='|')
print(f"High-quality entanglements: {len(result_step4)}")
result_step4.head()

## Step 5. Cluster redundant NCLEs into representative entanglements

Many entanglements are structurally degenerate variants of the same topological motif. This step collapses them into a non-redundant representative set.

> **`organism` parameter:** Sets the distance threshold optimized on a model organism.  
> Use `"Ecoli"` for *E. coli*, `"Human"` for *H. sapiens*, `"Yeast"` for *S. cerevisiae*.  
> For other organisms, supply `cut_off` directly to override the defaults.

**Output:** `$OUTDIR/Native_clustered_HQ_GE/1ZMR.csv`  
The CSV contains representative entanglements (one row per unique NCLE topology) plus associated degenerate loop variants.

In [ ]:
from EntDetect.clustering import ClusterNativeEntanglements

native_clustered_HQ_outdir = f"{OUTDIR}/Native_clustered_HQ_GE"
outfile   = "1ZMR.csv"

# Point to Step 4 output
NCLE_file = f"{OUTDIR}/Native_HQ_GE/1ZMR.csv"

clustering = ClusterNativeEntanglements(organism="Ecoli", cut_off=None)
clustering.Cluster_NativeEntanglements(
    NCLE_file,
    outdir=native_clustered_HQ_outdir,
    outfile=outfile,
    chain='A'
)

In [ ]:
result_step5 = pd.read_csv(f"{native_clustered_HQ_outdir}/1ZMR.csv", sep='|')
print(f"Representative entanglements: {len(result_step5)}")
result_step5.head()

## Step 6. Compute structural features for representative NCLEs

For each representative NCLE, computes:
- supercoiling characteristics of the entangled terminus;
- contact patterns between the loop and the rest of the structure;
- other structural descriptors capturing topological complexity.

**Output:** `$OUTDIR/Native_clustered_HQ_GE_features/P00558_1ZMR_A_uent_features.csv`

In [ ]:
from EntDetect.entanglement_features import FeatureGen

pdb                      = f"{DATASTORE}/user_input/reference_structures/1zmr_model_clean.pdb"
native_GQ_feature_outdir = f"{OUTDIR}/Native_clustered_HQ_GE_features"

# Point to Step 5 output
cluster_file = f"{OUTDIR}/Native_clustered_HQ_GE/1ZMR.csv"

FGen = FeatureGen(pdb, outdir=native_GQ_feature_outdir, cluster_file=cluster_file)

# gene  : UniProt accession for ecPGK
# chain : PDB chain identifier
# pdbid : four-letter PDB code (uppercase)
EntFeatures = FGen.get_uent_features(gene='P00558', chain='A', pdbid='1ZMR')
print(EntFeatures)

In [ ]:
print(f'Loading feature data from {native_GQ_feature_outdir}/P00558_1ZMR_A_uent_features.csv')
result_step6 = pd.read_csv(f"{native_GQ_feature_outdir}/P00558_1ZMR_A_uent_features.csv", sep='|')
print(f"Feature rows: {len(result_step6)}")
result_step6.head()

## Step 7. Visualize the representative entanglements

The interactive viewer below renders the 1ZMR structure directly in the notebook using **py3Dmol**.

| Colour | Meaning |
|--------|---------|
| Light grey | Rest of the chain |
| **Red** | Loop residues (`i` → `j`) for each representative NCLE |
| **Blue** + sphere | Crossing residues (`crossingsN` / `crossingsC`) |

For publication-quality figures, load the representative structures in **VMD** or **PyMOL** using the residue ranges identified here.

In [59]:
import py3Dmol
import pandas as pd
import json

# Load representative NCLEs
clustered_ncles = pd.read_csv(f"{native_clustered_HQ_outdir}/1ZMR.csv", sep='|').reset_index(drop=True)

ncles = []
for idx, row in clustered_ncles.iterrows():
    loop_start = int(row['i'])
    loop_end = int(row['j'])
    loop_sel = f"{loop_start}-{loop_end}"

    crossing_n = str(row['crossingsN']).strip()
    crossing_c = str(row['crossingsC']).strip()
    crossings = []
    if crossing_n.lower() != 'nan' and crossing_n:
        crossings.append(str(int(float(crossing_n.lstrip('+-')))))
    if crossing_c.lower() != 'nan' and crossing_c:
        crossings.append(str(int(float(crossing_c.lstrip('+-')))))

    cross_parts = []
    if crossings:
        if crossing_n.lower() != 'nan' and crossing_n:
            cross_parts.append(f"N-cross:{crossings[0]}")
        if crossing_c.lower() != 'nan' and crossing_c:
            cross_parts.append(f"C-cross:{crossings[-1]}")
    cross_label = ' | '.join(cross_parts) if cross_parts else 'no crossing'

    ncles.append({
        'label': f"NCLE {idx+1}: loop {loop_start}-{loop_end} | {cross_label}",
        'loop_start': loop_start,
        'loop_end': loop_end,
        'loop_sel': loop_sel,
        'cross_sel': ','.join(crossings),
    })

first_ncle = ncles[0]
print(f"loop_sel: {first_ncle['loop_sel']}")
print(f"crossing_sel: {first_ncle['cross_sel'] or 'none'}")

with open(pdb) as f:
    pdb_str = f.read()

# Working render path: full-structure zoom + first NCLE loop in red + crossing in blue.
view = py3Dmol.view(width=820, height=520)
view.addModel(pdb_str, 'pdb')
view.setStyle({'chain': 'A'}, {'cartoon': {'color': '#bfbfbf', 'opacity': 0.9}})
view.setStyle({'chain': 'A', 'resi': first_ncle['loop_sel']}, {'cartoon': {'color': '#E57373'}})
if first_ncle['cross_sel']:
    view.setStyle(
        {'chain': 'A', 'resi': first_ncle['cross_sel']},
        {'cartoon': {'color': '#1E88E5'}, 'sphere': {'color': '#1E88E5', 'radius': 0.7}},
    )
view.zoomTo({'chain': 'A'})
view.show()

# Reused by the dropdown controller cell below.
vname = f"viewer_{view.uniqueid}"
ncles_json = json.dumps(ncles)

print(
    f"Rendered base 1ZMR structure with first NCLE loop highlighted in red: "
    f"{first_ncle['loop_start']}-{first_ncle['loop_end']}"
)
print(f"Highlighted crossing residue in blue: {first_ncle['cross_sel'] or 'none'}")

from IPython.display import HTML, display

options_html = ''.join(
    f'<option value="{idx}">{ncle["label"]}</option>'
    for idx, ncle in enumerate(ncles)
)

display(HTML(f"""
<div style="font-family:sans-serif; margin-top:6px; padding:0 4px;">
  <label style="font-weight:600; margin-right:8px;">NCLE:</label>
  <select style="font-size:14px; padding:4px 8px; width:520px; cursor:pointer;"
          onchange="window._ncleSwitch(parseInt(this.value, 10))">
    {options_html}
  </select>
</div>
<script>
(function() {{
  var VNAME = "{vname}";
  var NCLES = {ncles_json};
  window._ncleSwitch = function(idx) {{
    var v = window[VNAME];
    if (!v) {{
      console.warn('viewer not found:', VNAME);
      return;
    }}
    var n = NCLES[idx];
    v.setStyle({{chain:"A"}}, {{cartoon:{{color:"#bfbfbf", opacity:0.9}}}});
    v.setStyle({{chain:"A", resi:n.loop_sel}}, {{cartoon:{{color:"#E57373"}}}});
    if (n.cross_sel) {{
      v.setStyle(
        {{chain:"A", resi:n.cross_sel}},
        {{cartoon:{{color:"#1E88E5"}}, sphere:{{color:"#1E88E5", radius:0.7}}}}
      );
    }}
    v.zoomTo({{chain:"A"}});
    v.render();
  }};
}})();
</script>
"""))

loop_sel: 228-274
crossing_sel: 221


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Rendered base 1ZMR structure with first NCLE loop highlighted in red: 228-274
Highlighted crossing residue in blue: 221


---

## Troubleshooting

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| `FileNotFoundError` on PDB | Wrong path | Verify the file exists under `$DATASTORE/user_input/reference_structures/` |
| Empty output file from Step 3 | Structure has no detectable NCLEs, or `g_threshold` too high | Lower `g_threshold` to 0.4 and rerun |
| Step 6 raises `KeyError` on gene or chain | `gene`, `chain`, or `pdbid` arguments do not match the structure | Check the PDB header for the correct chain ID |
| `ClusterNativeEntanglements` returns only 1 cluster | Normal for small proteins or a single entanglement topology | Proceed; the representative is the single detected NCLE |
| Viewer shows no colours | Residue numbers in CSV don't match PDB numbering | Check that the PDB was not renumbered after entanglement detection |